In [ ]:
from dflintdpy.data.config import HP
from dflintdpy.simulation.spni.config import build_run_config
from dflintdpy.simulation.spni.build import build_graph
from dflintdpy.models.graph import Graph

from pathlib import Path
import numpy as np

cwd = Path.cwd().parent


In [ ]:
cfg = HP()

run_cfg = build_run_config(
    cfg,
    load_real_world_graph=Path("../real_world_spni_data/transportation_networks/Anaheim_net.tntp"),
    source_node=108,
    target_node=410,
)

graph = build_graph(run_cfg)


In [ ]:
path_one_hot, objective = graph.solve()
path_edges = Graph.one_hot_to_arcs(graph, path_one_hot)

print("Objective:", objective)
print("Source node:", graph.source)
print("Target node:", graph.target)
print("Path edges:", path_edges)

%matplotlib inline
graph.visualize(colored_edges=path_one_hot)

In [ ]:
objective_values = np.full((len(graph.vertices), len(graph.vertices)), -1.0)
for source in range(len(graph.vertices)):
    for target in range(len(graph.vertices)):
        if source == target:
            continue
        run_cfg = build_run_config(
            cfg,
            load_real_world_graph=cwd / "real_world_spni_data/town_level_arcs.csv",
            source_node=source,
            target_node=target,
        )

        graph = build_graph(run_cfg)
        try:
            path_one_hot, objective = graph.solve()
        except:
            continue
        objective_values[source,target] = objective
    print(objective_values[source,:])

In [ ]:
objective_values.max()

In [ ]:
%matplotlib inline

from pathlib import Path
import matplotlib.pyplot as plt

from dflintdpy.utils.real_world_spni_data_handling import tntp_to_graph

graph_path = Path("../real_world_spni_data/transportation_networks/Anaheim_net.tntp")
graph = tntp_to_graph(graph_path)

graph.setObj(graph.cost, source=230, target=23)
path_mask, objective = graph.solve()

len(graph.vertices), len(graph.arcs), objective


In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

graph.visualize(
    colored_edges=path_mask,
    ax=ax,
    title="Anaheim TNTP Graph: Example Shortest Path",
    width=0.7,
    arrowsize=8,
    node_size=40,  # this is forwarded only if Graph.visualize later supports it
)

plt.show()
